# 💎 기능 4. 연구 스페이스 (Gems) - 업로드 파일 RAG 파이프라인 (기초)

본 가이드는 **Paper Agent** 프로젝트의 **연구 스페이스 (Gems) 기능** 중 **사용자가 업로드한 자료 파일로부터 텍스트를 추출하고, 청킹(Chunking)하여 벡터 DB에 등록 및 검색하는 핵심 원리**를 다룹니다.

학습 집중에 불필요한 예외 처리 및 Mock 폴백 코드를 제거하여, **실제 파일 RAG 핵심 구현 코드(`_extract_text`, `_chunk_text`, `GemFileRag` 클래스)의 핵심 로직만 간결하고 명확하게 확인**할 수 있도록 수정되었습니다.

---

## 📌 기초 학습 핵심 포인트
1. **백엔드 환경 설정 로드**
   - `backend/.env` 환경 변수와 패키지 경로를 주입합니다.
2. **파일 형식별 텍스트 추출 로직 (`_extract_text`)**
   - `pdfplumber`를 이용해 PDF 파일의 바이너리로부터 텍스트를 복원하고, `python-docx`를 이용해 Word 파일의 텍스트를 추출하는 실제 백엔드 소스 코드를 구현하고 테스트합니다.
3. **고정 크기 청킹 로직 (`_chunk_text`)**
   - `CHUNK_SIZE = 800` 자 및 `CHUNK_OVERLAP = 150` 자 기준으로 긴 텍스트를 중첩 분할하는 원리를 코드로 실습합니다.
4. **벡터 데이터베이스 연동 클래스 (`GemFileRag`) 구현**
   - LangChain `PGVector`와 연계하여 컬렉션을 생성하고 임베딩을 저장/검색하는 데이터베이스 핸들링 코드를 직접 구현합니다.

## 1단계. 백엔드 환경 설정 및 모듈 로드

노트북 실행 위치를 파악하고 백엔드 패키지 경로를 주입한 뒤, 필수 환경변수들을 로드합니다.

In [13]:
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

# Pydantic v2와 LangChain 연동 시 내부 직렬화 과정에서 발생하는 무해한 UserWarning 무시
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

# 현재 notebooks/gem_service/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

print(f"🔑 OpenAI API Key 설정 여부: {'설정됨(Yes)' if openai_key else '설정안됨(No)'}")
print(f"🗄️ Database URL: {database_url[:35]}..." if database_url else "🗄️ Database URL: 설정안됨")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
🔑 OpenAI API Key 설정 여부: 설정됨(Yes)
🗄️ Database URL: postgresql+asyncpg://postgres:postg...


## 2단계. 문서 파일 텍스트 추출 실습 (`_extract_text`)

실제 백엔드의 [file_rag.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/file_rag.py)에 구현된 파싱 함수 `_extract_text`를 정의합니다.
- **PDF 파일**: `pdfplumber`를 이용해 가상/실제 바이너리 스트림에서 다중 페이지 텍스트를 추출합니다.
- **Word 파일**: `docx.Document`를 이용해 문단별 텍스트를 조립합니다.
- **일반 텍스트**: UTF-8 디코딩을 시도하고 예외 시 Latin-1으로 복구합니다.

In [14]:
import io
import pdfplumber
from docx import Document as DocxDocument

def _extract_text(filename: str, file_bytes: bytes) -> str:
    """파일 형식에 따라 텍스트를 추출합니다.
    
    이 코드는 백엔드 api/v1/gems/file_rag.py의 내부 로직과 100% 동일합니다.
    """
    fname = filename.lower()

    # 1) PDF 파싱
    if fname.endswith(".pdf"):
        with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
            pages = [page.extract_text() or "" for page in pdf.pages]
        return "\n".join(p for p in pages if p.strip())

    # 2) Word 파일 파싱
    if fname.endswith(".docx") or fname.endswith(".doc"):
        doc = DocxDocument(io.BytesIO(file_bytes))
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

    # 3) TXT / MD / CSV 등 텍스트 계열
    try:
        return file_bytes.decode("utf-8")
    except UnicodeDecodeError:
        return file_bytes.decode("latin-1", errors="ignore")


# --- 실행 테스트 ---
# 1. 테스트용 Word (.docx) 바이너리 생성
def create_mock_docx() -> bytes:
    doc = DocxDocument()
    doc.add_heading("CRISPR Gene Editing for Cancer Therapy", level=1)
    doc.add_paragraph("This document discusses the application of CRISPR-Cas9 in targeting breast cancer oncogenes.")
    doc.add_paragraph("Recent studies show a significant reduction in tumor growth when targeting specific regulatory elements.")
    file_stream = io.BytesIO()
    doc.save(file_stream)
    return file_stream.getvalue()

docx_bytes = create_mock_docx()
extracted_docx = _extract_text("sample_paper.docx", docx_bytes)
print("✨ [DOCX 추출 결과]:\n", extracted_docx)
print("\n" + "="*50 + "\n")

# 2. 실제 폴더 내 astro.pdf 파일 읽기 테스트
pdf_file_path = current_dir / "astro.pdf"
if pdf_file_path.exists():
    pdf_bytes = pdf_file_path.read_bytes()
    extracted_pdf = _extract_text("astro.pdf", pdf_bytes)
    print("✨ [PDF 추출 결과 (일부)]:\n", extracted_pdf[:300] + "...")
else:
    print("⚠️ astro.pdf 파일이 존재하지 않아 PDF 추출을 생략합니다.")

✨ [DOCX 추출 결과]:
 CRISPR Gene Editing for Cancer Therapy
This document discusses the application of CRISPR-Cas9 in targeting breast cancer oncogenes.
Recent studies show a significant reduction in tumor growth when targeting specific regulatory elements.


✨ [PDF 추출 결과 (일부)]:
 A S T R O N O M Y
천문학 입문
태양계에서 우주까지
밤하늘의 별 하나에서 시작해 우리은하, 그리고 138억 년 우주
의 역사까지 —
처음 천문학을 만나는 이를 위한 한 권의 안내서.
학습용 개론서 · 전 10장 구성
Introduction to Astronomy
CONTENTS
목차
이 책의 구성
"우리는 별의 먼지로 이루어져 있다."
우리 몸을 이루는 원소 대부분은 오래전 별의 내부에서 만들어졌습니다. 천문학을 배운다는 것은
곧 우리 자신의 기원을 더듬어 가는 일이기도 합니다. 이 책은 가까운 태양계에서 출발해 점점 ...


## 3단계. 고정 크기 청킹 로직 실습 (`_chunk_text`)

긴 문장을 고정 크기 글자 수 단위(`CHUNK_SIZE = 800`, `CHUNK_OVERLAP = 150`)로 분할하여 조각(Chunk) 배열로 리턴하는 로직입니다. 
앞선 질문에서 확인한 바와 같이, 파이썬 문자열 슬라이싱을 이용한 **글자 수(Character count)** 기반 분할을 수행합니다.

In [15]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def _chunk_text(text: str) -> list[str]:
    """텍스트를 고정 크기 청크로 분할합니다 (CHUNK_OVERLAP 만큼 겹침).
    
    이 코드는 백엔드 api/v1/gems/file_rag.py의 내부 로직과 100% 동일합니다.
    """
    if not text.strip():
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - CHUNK_OVERLAP
    return chunks

# 테스트용 긴 텍스트 분할
sample_long_text = (
    "CRISPR-Cas9 technology has revolutionized the field of molecular biology. "
    "By using a guide RNA (gRNA) to target specific DNA sequences, researchers can introduce precise double-strand breaks. "
    "This allows for targeted gene knockouts or knock-ins. In oncology research, CRISPR is heavily used to model "
    "cancer mutations and screen for potential drug targets. Specifically, in breast cancer studies, target elements "
    "in the HER2 pathway are analyzed to overcome drug resistance. However, off-target effects remain a major challenge. "
    "Advanced variants like high-fidelity Cas9 and base editors are being developed to mitigate these risks. "
    "Furthermore, delivery systems such as lipid nanoparticles (LNPs) and adeno-associated viruses (AAVs) are optimized "
    "for clinical trials. The integration of CRISPR with single-cell RNA sequencing allows for high-throughput screens "
    "revealing complex cellular phenotypes. Thus, gene editing continues to pave the way for personalized medicine."
)

chunks = _chunk_text(sample_long_text)
print(f"📋 총 생성된 청크 수: {len(chunks)}개")
for idx, chunk in enumerate(chunks):
    metadata = {"filename": "crispr_review.docx", "chunk_index": idx}
    print(f"\n[Chunk {idx}] (Metadata: {metadata})")
    print(f"내용: {chunk[:120]}... [길이: {len(chunk)}자]")

📋 총 생성된 청크 수: 2개

[Chunk 0] (Metadata: {'filename': 'crispr_review.docx', 'chunk_index': 0})
내용: CRISPR-Cas9 technology has revolutionized the field of molecular biology. By using a guide RNA (gRNA) to target specific... [길이: 800자]

[Chunk 1] (Metadata: {'filename': 'crispr_review.docx', 'chunk_index': 1})
내용: ery systems such as lipid nanoparticles (LNPs) and adeno-associated viruses (AAVs) are optimized for clinical trials. Th... [길이: 321자]


## 4단계. pgvector 연동 및 유사도 검색 실습 (`GemFileRag` 구현)

실제 백엔드의 RAG 검색 핵심 컨트롤러인 `GemFileRag` 클래스의 구현을 정의합니다.
이 클래스는 OpenAI Large 임베딩 엔진과 지정된 데이터베이스 URL 커넥션을 활용해 `gem_{gem_id}_files` 컬렉션을 개설하고 임베딩 데이터의 등록(`process_and_store`), 유사도 검색(`search`), 그리고 삭제(`delete_collection`)를 총괄합니다.

여기서는 실제 데이터베이스와 OpenAI API를 통해 동작하는 실제 코드만을 단순하고 직관적으로 실행합니다.

In [16]:
from langchain.embeddings import init_embeddings
from langchain_postgres import PGVector
import logging

logger = logging.getLogger("notebook_gem_service")

def _collection_name(gem_id: str) -> str:
    return f"gem_{gem_id}_files"

class GemFileRag:
    """Gem별 업로드 파일의 RAG 파이프라인을 관리하는 코어 클래스."""
    def __init__(self, connection_url: str = ""):
        self.connection = connection_url
        self._embeddings = None

    def get_embeddings(self):
        if self._embeddings is None:
            self._embeddings = init_embeddings(model="openai:text-embedding-3-large")
        return self._embeddings

    def _get_vectorstore(self, gem_id: str) -> PGVector:
        return PGVector(
            embeddings=self.get_embeddings(),
            collection_name=_collection_name(gem_id),
            connection=self.connection,
            async_mode=True,
        )

    async def process_and_store(self, gem_id: str, filename: str, file_bytes: bytes) -> int:
        text = _extract_text(filename, file_bytes)
        chunks = _chunk_text(text)

        if not chunks:
            return 0

        vectorstore = self._get_vectorstore(gem_id)
        metadatas = [{"filename": filename, "chunk_index": i} for i in range(len(chunks))]
        await vectorstore.aadd_texts(chunks, metadatas=metadatas)
        print(f"✅ [Vector DB] {filename} → {len(chunks)}개 청크 저장 완료")
        return len(chunks)

    async def search(self, gem_id: str, query: str, k: int = 3) -> list[dict]:
        vectorstore = self._get_vectorstore(gem_id)
        results = await vectorstore.asimilarity_search_with_score(query, k=k)

        return [
            {
                "filename": (doc.metadata or {}).get("filename", ""),
                "chunk_index": (doc.metadata or {}).get("chunk_index", 0),
                "text_chunk": doc.page_content,
                "score": round(1.0 - score, 4),
            }
            for doc, score in results
        ]

    async def delete_collection(self, gem_id: str) -> None:
        vectorstore = self._get_vectorstore(gem_id)
        await vectorstore.adelete_collection()
        print(f"🧹 [Vector DB] 컬렉션 {_collection_name(gem_id)} 삭제 완료")


# --- 실행 테스트 ---
test_gem_id = "demo-gem-uuid-12345"
gem_file_rag = GemFileRag(connection_url=database_url.replace("postgresql+asyncpg://", "postgresql+psycopg_async://"))

filename = "gem_test_paper.txt"
file_content = (
    "This is a specific study on exoplanetary atmospheres. "
    "We present detections of water vapor in the atmosphere of the hot Jupiter planet HD 189733b. "
    "Observations were carried out using space-based spectroscopy."
)
file_bytes = file_content.encode("utf-8")

# 1. 문서 등록
print("🚀 [Gems RAG] 1. 파일 적재 시도...")
await gem_file_rag.process_and_store(test_gem_id, filename, file_bytes)

# 2. RAG 검색
query = "water vapor in exoplanet HD 189733b"
print(f"\n🔎 [Gems RAG] 2. 유사도 검색 수행 | 검색어: '{query}'")
search_results = await gem_file_rag.search(test_gem_id, query, k=1)
    
print(f"📋 검색된 청크 목록 (총 {len(search_results)}개):")
for idx, r in enumerate(search_results, 1):
    print(f"  [{idx}] 파일명: {r['filename']} (청크 인덱스: {r['chunk_index']})")
    print(f"      유사도 점수(Score): {r['score']}")
    print(f"      추출 내용: \"{r['text_chunk']}\"")

# 3. 컬렉션 삭제
print(f"\n🧹 [Gems RAG] 3. 컬렉션 삭제 청소...")
await gem_file_rag.delete_collection(test_gem_id)

🚀 [Gems RAG] 1. 파일 적재 시도...
✅ [Vector DB] gem_test_paper.txt → 1개 청크 저장 완료

🔎 [Gems RAG] 2. 유사도 검색 수행 | 검색어: 'water vapor in exoplanet HD 189733b'
📋 검색된 청크 목록 (총 1개):
  [1] 파일명: gem_test_paper.txt (청크 인덱스: 0)
      유사도 점수(Score): 0.7039
      추출 내용: "This is a specific study on exoplanetary atmospheres. We present detections of water vapor in the atmosphere of the hot Jupiter planet HD 189733b. Observations were carried out using space-based spectroscopy."

🧹 [Gems RAG] 3. 컬렉션 삭제 청소...
🧹 [Vector DB] 컬렉션 gem_demo-gem-uuid-12345_files 삭제 완료


## 5단계. 백엔드 구현 파일 연계 정보

이 노트북에서 직접 타이핑하고 실습해 본 파일 처리 및 벡터 적재 기능은 실제 백엔드의 다음 소스파일들에 물리적으로 격리 및 구현되어 있습니다:

1. **`api.v1.gems.file_rag`**
   - [file_rag.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/file_rag.py): `pdfplumber` 및 `docx` 연동 텍스트 추출 함수(`_extract_text`), 청킹 함수(`_chunk_text`), pgvector 벡터 DB 적재 및 비동기 유사도 검색(`GemFileRag`) 클래스 전체가 이 곳에 구현되어 있습니다.
2. **`api.v1.gems.services`**
   - [services.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/services.py): 사용자가 업로드한 파일을 저장할 때 `gem_file_rag.process_and_store`를 호출하여 DB 적재를 시작하고, 업로드된 파일 정보의 목록을 DB 테이블(`gem_files`)에 적재해 관리합니다.
3. **`api.v1.gems.endpoints`**
   - [endpoints.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/gems/endpoints.py): `/gems/{gem_id}/files` 파일 업로드 API 엔드포인트가 노출되어 있으며, 프론트엔드 통신 모듈과 직접 연결됩니다.